In [1]:
import glob
import polars
import json
import taxburst.checks
from collections import defaultdict

In [2]:

taxinfo_ranks = [
    "superkingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "strain",
    "genome",
]

In [3]:
ROOT='/home/ctbrown/scratch3/2025-jean-binning-loss/'

In [4]:
SAMPLES = [ x.strip() for x in open(ROOT + 'SAMPLE-LIST.BIG.txt') ]
print(f"loaded {len(SAMPLES)} accessions")

# subset for testing                                                            
SAMPLES = SAMPLES[:100]
print(f"subsetting to {len(SAMPLES)} total.")

loaded 915 accessions
subsetting to 100 total.


In [5]:
bin_tax = {}
contig_tax = {}
read_tax = {}

for n, sample in enumerate(SAMPLES):
    if n % 10 == 0:
        print('...', n, 'of', len(SAMPLES))
    with open(ROOT + f'outputs/tax/{sample}.bins.gather.json') as fp:
        bin_tax[sample] = json.load(fp)
    with open(ROOT + f'outputs/tax/{sample}.reads.gather.json') as fp:
        read_tax[sample] = json.load(fp)
    with open(ROOT + f'outputs/tax/{sample}.contigs.gather.json') as fp:
        contig_tax[sample] = json.load(fp)


... 0 of 100
... 10 of 100
... 20 of 100
... 30 of 100
... 40 of 100
... 50 of 100
... 60 of 100
... 70 of 100
... 80 of 100
... 90 of 100


In [6]:
def extract_nodes_by_rank_and_name(tree):
    nodes = taxburst.checks.collect_all_nodes(tree)
    nodes_by_rank = defaultdict(list)
    nodes_by_name = {}
    for node in nodes:
        rank = node["rank"]
        nodes_by_rank[rank].append(node)
        name = node["name"]
        assert name not in nodes_by_name # names should be unique w/in tree
        nodes_by_name[name] = node
    return nodes_by_rank, nodes_by_name

by_rank, by_name = extract_nodes_by_rank_and_name(bin_tax[sample])
print(len(by_rank))

8


In [7]:
for sample in SAMPLES:
    bin_by_rank, bin_by_name = extract_nodes_by_rank_and_name(bin_tax[sample])
    contig_by_rank, contig_by_name = extract_nodes_by_rank_and_name(contig_tax[sample])
    read_by_rank, read_by_name = extract_nodes_by_rank_and_name(read_tax[sample])

    print(f"sample: {sample}")

    rank = 'superkingdom'
    for read_node in read_by_rank[rank]:
        name = read_node["name"]
        if name == 'unclassified':
            continue
        
        contig_node = contig_by_name.get(name)
        bin_node = bin_by_name.get(name)
    
        read_count = read_node["count"]
        contig_count = 0
        if contig_node:
            contig_count = contig_node["count"]
        bin_count = 0
        if bin_node:
            bin_count = bin_node["count"]
    
        contig_p = contig_count / read_count * 100
        bin_p = bin_count / read_count * 100
        print(f" {name:<30} {read_count:>10.0f} {contig_count:>10.0f} ({contig_p:<4.1f}%) {bin_count:>10.0f} ({bin_p:<4.1f}%)")

sample: ERR1135212
 d__Bacteria                       2154254    1325229 (61.5%)     360802 (16.7%)
 d__Archaea                          11419       3216 (28.2%)          0 (0.0 %)
sample: ERR1135219
 d__Bacteria                       2332725    1415363 (60.7%)     359550 (15.4%)
 d__Archaea                           4515       1503 (33.3%)          0 (0.0 %)
sample: ERR1135259
 d__Bacteria                       1675015    1005178 (60.0%)     226270 (13.5%)
 d__Archaea                          19083      11311 (59.3%)       8228 (43.1%)
sample: ERR1135262
 d__Bacteria                       1928356    1244734 (64.5%)     353983 (18.4%)
 d__Archaea                           7269       1279 (17.6%)          0 (0.0 %)
sample: ERR1135273
 d__Bacteria                       3612850    2786581 (77.1%)     787835 (21.8%)
 d__Archaea                           4298       1603 (37.3%)          0 (0.0 %)
sample: ERR1135298
 d__Bacteria                       2241856    1203433 (53.7%)     355090 (15

In [8]:
for sample in SAMPLES:
    bin_by_rank, bin_by_name = extract_nodes_by_rank_and_name(bin_tax[sample])
    contig_by_rank, contig_by_name = extract_nodes_by_rank_and_name(contig_tax[sample])
    read_by_rank, read_by_name = extract_nodes_by_rank_and_name(read_tax[sample])

    print(f"sample: {sample}")

    rank = 'phylum'
    for read_node in read_by_rank[rank]:
        name = read_node["name"]
        if name == 'unclassified':
            continue
        
        contig_node = contig_by_name.get(name)
        bin_node = bin_by_name.get(name)
    
        read_count = read_node["count"]
        contig_count = 0
        if contig_node:
            contig_count = contig_node["count"]
        bin_count = 0
        if bin_node:
            bin_count = bin_node["count"]
    
        contig_p = contig_count / read_count * 100
        bin_p = bin_count / read_count * 100
        print(f" {name:<30} {read_count:>10.0f} {contig_count:>10.0f} ({contig_p:<4.1f}%) {bin_count:>10.0f} ({bin_p:<4.1f}%)")

sample: ERR1135212
 p__Bacteroidota                   1054336     813974 (77.2%)     160621 (15.2%)
 p__Spirochaetota                   120325      78733 (65.4%)      20292 (16.9%)
 p__Verrucomicrobiota                79356      50045 (63.1%)      13522 (17.0%)
 p__Planctomycetota                   7175       2988 (41.6%)          6 (0.1 %)
 p__Bacillota                       829339     359444 (43.3%)     156286 (18.8%)
 p__Fibrobacterota                    9225       1433 (15.5%)         29 (0.3 %)
 p__Desulfobacterota_I                5782       1385 (24.0%)          0 (0.0 %)
 p__Cyanobacteriota                  17374      11106 (63.9%)       9942 (57.2%)
 p__Pseudomonadota                   14138       2362 (16.7%)         37 (0.3 %)
 p__Campylobacterota                  5657       1045 (18.5%)          0 (0.0 %)
 p__Elusimicrobiota                   3834       1706 (44.5%)          5 (0.1 %)
 p__Myxococcota                       1179         24 (2.0 %)          5 (0.4 %)
 p__Actin